In [1]:
from typing import *
from langgraph.graph import StateGraph, START, END


In [2]:
class AgentState(TypedDict):
    number1 : int
    number2 : int
    operation : str
    result : int

In [ ]:
def adderNode(state : AgentState) -> AgentState:
    """
    This node adds two numbers
    """
    state["result"] = state["number1"] + state["number2"]
    return state
    
def subtractNode(state : AgentState) -> AgentState:
    """
    This node subtracts two numbers
    """
    state["result"] = state["number1"] - state["number2"]
    return state


def decisionNode(state : AgentState) -> AgentState:
    """
    This node decides the operation to be performed.
    We need to return the edge name as a string, instead of the node name. 
    """
    if state["operation"] == "+":
        return "additionEdge"
    elif state["operation"] == "-":
        return "subtractionEdge"
    else:
        raise ValueError(f"Invalid operation: {state['operation']}")



In [16]:
graph = StateGraph(AgentState)

graph.add_node("adderNode", adderNode)
graph.add_node("subtractNode", subtractNode)
graph.add_node("router", lambda state:state)
graph.add_edge(START, "router")

graph.add_conditional_edges(
    "router",
    decisionNode,
    {   # Edge:Node
        "additionEdge" : "adderNode",
        "subtractionEdge" : "subtractNode"
    }
)


graph.add_edge("adderNode", END)
graph.add_edge("subtractNode", END)


In [18]:
app = graph.compile()

agent_state1 = {"number1" : 10, "number2" : 20, "operation" : "+"}
agent_state2 = {"number1" : 10, "number2" : 20, "operation" : "-"}

result1 = app.invoke(agent_state1)
result2 = app.invoke(agent_state2)

print(result1)
print(result2)

{'number1': 10, 'number2': 20, 'operation': '+', 'result': 30}
{'number1': 10, 'number2': 20, 'operation': '-', 'result': -10}
